In [1]:
import pandas as pd
import numpy as np
import keras
from keras import layers, ops
from sklearn.model_selection import KFold

import ast

In [2]:
def get_fixed_interval_params():
    """
    Fixed interval vocabulary covering [-48, +48] semitones (4 octaves).

    This avoids any data leakage between folds and matches IDyOM's
    approach where the viewpoint alphabet is defined by the domain,
    not by the data.
    """
    min_interval = -48
    max_interval = 48
    # = 49, so 0 stays PAD
    interval_offset = -min_interval + 1
    interval_vocab_size = max_interval - min_interval + 2

    return interval_offset, interval_vocab_size

In [3]:
def prepare_melodies(melodies, window_size=16, pitch_to_idx=None,
                            interval_offset=None, interval_vocab_size=None):
    """
    Sliding-window examples with five input features matching IDyOM's
    best viewpoint system: (CPINT (CPCINT CPINTFIP) (CPITCH CPINTFREF)
    """
    if pitch_to_idx is None:
        all_pitches = sorted(set(p for mel in melodies for p in mel))
        pitch_to_idx = {p: i + 1 for i, p in enumerate(all_pitches)}

    idx_to_pitch = {i: p for p, i in pitch_to_idx.items()}
    vocab_size = max(pitch_to_idx.values()) + 1
    PAD = 0

    if interval_offset is None or interval_vocab_size is None:
        interval_offset, interval_vocab_size = get_fixed_interval_params()

    # cpcint: interval mod 12, range [0, 11] → shifted to 1-12, 0=PAD
    CPCINT_VOCAB_SIZE = 13  # PAD + 12 pitch classes

    xs_pitch = []
    xs_cpint = []
    xs_cpcint = []
    xs_cpintfip = []
    xs_cpintfref = []
    ys = []
    skipped = 0

    for mel in melodies:
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]
        raw = [p for p in mel if p in pitch_to_idx]

        if len(indexed) < 2:
            skipped += 1
            continue

        first_pitch = raw[0]

        for t in range(1, len(indexed)):
            start = max(0, t - window_size)
            context = indexed[start:t]
            raw_context = raw[start:t]

            # cpint: interval from previous note
            raw_cpint = [0]
            for i in range(1, len(raw_context)):
                raw_cpint.append(raw_context[i] - raw_context[i - 1])

            # cpcint: cpint mod 12 (pitch-class interval)
            raw_cpcint = [0]
            for i in range(1, len(raw_context)):
                raw_cpcint.append((raw_context[i] - raw_context[i - 1]) % 12)

            # cpintfip: interval from first pitch of piece
            raw_cpintfip = [p - first_pitch for p in raw_context]

            # cpintfref: interval from referent
            # Without key annotations, this equals cpintfip
            raw_cpintfref = [p - first_pitch for p in raw_context]

            # Left-pad
            pad_len = window_size - len(context)

            pitch_seq = [PAD] * pad_len + context

            cpint_seq = [PAD] * pad_len
            for iv in raw_cpint:
                shifted = iv + interval_offset
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                cpint_seq.append(shifted)

            cpcint_seq = [PAD] * pad_len
            for iv in raw_cpcint:
                cpcint_seq.append(iv + 1)  # shift so 0=PAD, 1-12=intervals

            cpintfip_seq = [PAD] * pad_len
            for iv in raw_cpintfip:
                shifted = iv + interval_offset
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                cpintfip_seq.append(shifted)

            cpintfref_seq = [PAD] * pad_len
            for iv in raw_cpintfref:
                shifted = iv + interval_offset
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                cpintfref_seq.append(shifted)

            xs_pitch.append(pitch_seq)
            xs_cpint.append(cpint_seq)
            xs_cpcint.append(cpcint_seq)
            xs_cpintfip.append(cpintfip_seq)
            xs_cpintfref.append(cpintfref_seq)
            ys.append(indexed[t])

    if skipped > 0:
        print(f"  Warning: skipped {skipped} melodies")

    print(f"  Total training examples: {len(xs_pitch)} (from {len(melodies)} melodies)")

    return (np.array(xs_pitch), np.array(xs_cpint), np.array(xs_cpcint),
            np.array(xs_cpintfip), np.array(xs_cpintfref), np.array(ys),
            vocab_size, pitch_to_idx, idx_to_pitch)

Note that if cpintfip and cpintfref are identical for your data (no key annotations), we would then want to drop one to avoid redundancy.

In order to check the effect we can run IDyOM first with cpintfip alone and cpintfref alone and see if they give different ICs.

If they're the same, we then drop cpintfref and use only 4 inputs.

In [4]:
# Model components
class TokenAndPositionEmbedding(layers.Layer):
    """Embeds tokens and adds learned positional encoding."""

    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim, mask_zero=False
        )
        self.pos_emb = layers.Embedding(
            input_dim=max_len, output_dim=embed_dim
        )

    def call(self, x):
        seq_len = ops.shape(x)[-1]
        positions = ops.arange(start=0, stop=seq_len, step=1)
        token_embeddings = self.token_emb(x)
        position_embeddings = self.pos_emb(positions)
        return token_embeddings + position_embeddings


class SlidingWindowTransformerBlock(layers.Layer):
    """
    Transformer block where position i can only attend to
    positions [i-window_size+1, ..., i].
    """

    def __init__(self, embed_dim, num_heads, ff_dim, window_size, dropout_rate=0.1):
        super().__init__()
        self.window_size = window_size
        self.att = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads
        )
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)

    def _sliding_window_mask(self, seq_len):
        """Create a causal sliding window attention mask."""
        causal = ops.tril(ops.ones((seq_len, seq_len), dtype="bool"))
        rows = ops.arange(seq_len)[:, None]
        cols = ops.arange(seq_len)[None, :]
        window = (rows - cols) < self.window_size
        mask = ops.cast(causal & window, dtype="bool")
        return mask

    def call(self, inputs, training=False):
        seq_len = ops.shape(inputs)[1]
        mask = self._sliding_window_mask(seq_len)

        attn_output = self.att(
            inputs, inputs, attention_mask=mask
        )
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

In [5]:
def build_transformer(
    vocab_size,
    interval_vocab_size,
    cpcint_vocab_size=13,
    window_size=16,
    embed_dim=64,
    num_heads=4,
    ff_dim=128,
    num_layers=2,
    dropout_rate=0.1,
):
    """
    Transformer with five input streams matching IDyOM's best viewpoint
    system: (CPINT (CPCINT CPINTFIP) (CPITCH CPINTFREF)).
    Each viewpoint gets its own embedding.
    """
    pitch_input = keras.Input(shape=(window_size,), name="pitch")
    cpint_input = keras.Input(shape=(window_size,), name="cpint")
    cpcint_input = keras.Input(shape=(window_size,), name="cpcint")
    cpintfip_input = keras.Input(shape=(window_size,), name="cpintfip")
    cpintfref_input = keras.Input(shape=(window_size,), name="cpintfref")

    # Pitch embedding with positional encoding
    pitch_emb = TokenAndPositionEmbedding(window_size, vocab_size, embed_dim)(pitch_input)

    # Viewpoint embeddings (no positional encoding — position in pitch stream)
    cpint_emb = layers.Embedding(interval_vocab_size, embed_dim)(cpint_input)
    cpcint_emb = layers.Embedding(cpcint_vocab_size, embed_dim)(cpcint_input)
    cpintfip_emb = layers.Embedding(interval_vocab_size, embed_dim)(cpintfip_input)
    cpintfref_emb = layers.Embedding(interval_vocab_size, embed_dim)(cpintfref_input)

    # Combine all streams
    x = layers.Add()([pitch_emb, cpint_emb, cpcint_emb, cpintfip_emb, cpintfref_emb])

    for _ in range(num_layers):
        x = SlidingWindowTransformerBlock(
            embed_dim, num_heads, ff_dim, window_size, dropout_rate
        )(x)

    x = x[:, -1, :]
    outputs = layers.Dense(vocab_size, activation="softmax")(x)

    model = keras.Model(
        inputs=[pitch_input, cpint_input, cpcint_input, cpintfip_input, cpintfref_input],
        outputs=outputs,
    )
    return model

In [6]:
def compute_ic(model, xs_pitch, xs_cpint, xs_cpcint, xs_cpintfip, xs_cpintfref, ys):
    """
    Compute Information Content (IC) in bits for each note prediction.
    """
    probs = model.predict(
        [xs_pitch, xs_cpint, xs_cpcint, xs_cpintfip, xs_cpintfref], verbose=0
    )

    all_ics = []
    for i in range(len(xs_pitch)):
        p = probs[i][ys[i]]
        p = max(p, 1e-10)
        all_ics.append(-np.log2(p))

    return np.mean(all_ics), all_ics


def compute_ic_per_melody(model, melodies, window_size, pitch_to_idx,
                          interval_offset, interval_vocab_size):
    """
    Compute mean IC per melody.
    """
    PAD = 0
    melody_ics = []

    for mel_idx, mel in enumerate(melodies):
        indexed = [pitch_to_idx[p] for p in mel if p in pitch_to_idx]
        raw = [p for p in mel if p in pitch_to_idx]
        if len(indexed) < 2:
            continue

        first_pitch = raw[0]

        mel_xs_pitch = []
        mel_xs_cpint = []
        mel_xs_cpcint = []
        mel_xs_cpintfip = []
        mel_xs_cpintfref = []
        mel_ys = []

        for t in range(1, len(indexed)):
            start = max(0, t - window_size)
            context = indexed[start:t]
            raw_context = raw[start:t]

            # cpint
            raw_cpint = [0]
            for i in range(1, len(raw_context)):
                raw_cpint.append(raw_context[i] - raw_context[i - 1])

            # cpcint
            raw_cpcint = [0]
            for i in range(1, len(raw_context)):
                raw_cpcint.append((raw_context[i] - raw_context[i - 1]) % 12)

            # cpintfip
            raw_cpintfip = [p - first_pitch for p in raw_context]

            # cpintfref (same as cpintfip without key annotations)
            raw_cpintfref = [p - first_pitch for p in raw_context]

            # Left-pad
            pad_len = window_size - len(context)

            pitch_seq = [PAD] * pad_len + context

            cpint_seq = [PAD] * pad_len
            for iv in raw_cpint:
                shifted = iv + interval_offset
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                cpint_seq.append(shifted)

            cpcint_seq = [PAD] * pad_len
            for iv in raw_cpcint:
                cpcint_seq.append(iv + 1)

            cpintfip_seq = [PAD] * pad_len
            for iv in raw_cpintfip:
                shifted = iv + interval_offset
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                cpintfip_seq.append(shifted)

            cpintfref_seq = [PAD] * pad_len
            for iv in raw_cpintfref:
                shifted = iv + interval_offset
                shifted = max(1, min(shifted, interval_vocab_size - 1))
                cpintfref_seq.append(shifted)

            mel_xs_pitch.append(pitch_seq)
            mel_xs_cpint.append(cpint_seq)
            mel_xs_cpcint.append(cpcint_seq)
            mel_xs_cpintfip.append(cpintfip_seq)
            mel_xs_cpintfref.append(cpintfref_seq)
            mel_ys.append(indexed[t])

        mel_xs_pitch = np.array(mel_xs_pitch)
        mel_xs_cpint = np.array(mel_xs_cpint)
        mel_xs_cpcint = np.array(mel_xs_cpcint)
        mel_xs_cpintfip = np.array(mel_xs_cpintfip)
        mel_xs_cpintfref = np.array(mel_xs_cpintfref)
        mel_ys = np.array(mel_ys)

        probs = model.predict(
            [mel_xs_pitch, mel_xs_cpint, mel_xs_cpcint,
             mel_xs_cpintfip, mel_xs_cpintfref], verbose=0
        )

        ics = []
        for i in range(len(mel_xs_pitch)):
            p = max(probs[i][mel_ys[i]], 1e-10)
            ics.append(-np.log2(p))

        melody_ics.append({
            "melody_idx": mel_idx,
            "mean_ic": np.mean(ics),
            "n_notes": len(ics),
        })

    return melody_ics

In [8]:
def train_model(
    melodies,
    window_size=10,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    validation_split=0.1,
    patience=5,
):
    """
    Simple full-corpus training (for initial testing only).
    For proper evaluation, use run_kfold.
    """
    # Fixed interval parameters to avoid data leakage
    interval_offset, interval_vocab_size = get_fixed_interval_params()

    xs_pitch, xs_cpint, xs_cpcint, xs_cpintfip, xs_cpintfref, \
    ys, vocab_size, pitch_to_idx, idx_to_pitch = \
        prepare_melodies(
            melodies=melodies,
            window_size=window_size,
            interval_offset=interval_offset,
            interval_vocab_size=interval_vocab_size,
            )

    print(f"Data prepared:")
    print(f"  Melodies: {len(melodies)}")
    print(f"  Vocab size: {vocab_size} ({vocab_size - 1} pitches + PAD)")
    print(f"  Window size: {window_size}")
    print(f"  Training examples: {xs_pitch.shape[0]}")

    model = build_transformer(
        vocab_size=vocab_size,
        window_size=window_size,
        embed_dim=embed_dim,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_layers=num_layers,
        dropout_rate=dropout_rate,
    )

    print(f"\n  Total parameters: {model.count_params():,}")
    model.summary()

    model.compile(
        loss=keras.losses.SparseCategoricalCrossentropy(),
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    )

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=patience, restore_best_weights=True,
    )

    history = model.fit(
        [xs_pitch, xs_cpint, xs_cpcint, xs_cpintfip, xs_cpintfref], ys,
        batch_size=batch_size, epochs=epochs,
        validation_split=validation_split, callbacks=[early_stop],
    )

    return model, history, {
        "vocab_size": vocab_size,
        "interval_vocab_size": interval_vocab_size,
        "interval_offset": interval_offset,
        "pitch_to_idx": pitch_to_idx,
        "idx_to_pitch": idx_to_pitch,
        "window_size": window_size,
    }

In [9]:
def run_kfold(
    melodies,
    melody_ids=None,
    k=10,
    window_size=16,
    embed_dim=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout_rate=0.1,
    batch_size=32,
    epochs=50,
    patience=5,
    random_state=42,
    export_folds_path=None,
):
    """
    K-fold cross-validation for fair comparison with IDyOM.
    """
    if melody_ids is None:
        melody_ids = list(range(len(melodies)))

    interval_offset, interval_vocab_size = get_fixed_interval_params()

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)

    all_ics = []
    fold_results = []
    fold_assignments = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(melodies)):
        print(f"\n{'='*60}")
        print(f"FOLD {fold + 1}/{k}")
        print(f"{'='*60}")

        train_melodies = [melodies[i] for i in train_idx]
        test_melodies = [melodies[i] for i in test_idx]
        train_ids = [melody_ids[i] for i in train_idx]
        test_ids = [melody_ids[i] for i in test_idx]

        print(f"Train: {len(train_melodies)} melodies")
        print(f"Test:  {len(test_melodies)} melodies")

        # Record fold assignments
        for idx in train_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "train",
            })
        for idx in test_idx:
            fold_assignments.append({
                "melody_id": melody_ids[idx],
                "fold": fold + 1,
                "split": "test",
            })

        # Prepare data: vocabulary built from training set only
        xs_pitch_train, xs_cpint_train, xs_cpcint_train, xs_cpintfip_train, \
        xs_cpintfref_train, ys_train, vocab_size, pitch_to_idx, _ = \
            prepare_melodies(
                melodies=train_melodies,
                window_size=window_size,
                interval_offset=interval_offset,
                interval_vocab_size=interval_vocab_size,
            )

        # Test data uses same vocabulary
        xs_pitch_test, xs_cpint_test, xs_cpcint_test, xs_cpintfip_test, \
        xs_cpintfref_test, ys_test, _, _, _ = \
            prepare_melodies(
                melodies=test_melodies,
                window_size=window_size,
                pitch_to_idx=pitch_to_idx,
                interval_offset=interval_offset,
                interval_vocab_size=interval_vocab_size,
            )

        print(f"Vocab size: {vocab_size}")
        print(f"Interval vocab size: {interval_vocab_size}")

        # Build and train
        model = build_transformer(
            vocab_size=vocab_size,
            interval_vocab_size=interval_vocab_size,
            window_size=window_size,
            embed_dim=embed_dim,
            num_heads=num_heads,
            ff_dim=ff_dim,
            num_layers=num_layers,
            dropout_rate=dropout_rate,
        )

        if fold == 0:
            print(f"Total parameters: {model.count_params():,}")

        model.compile(
            loss=keras.losses.SparseCategoricalCrossentropy(),
            optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        )

        early_stop = keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )

        model.fit(
            [xs_pitch_train, xs_cpint_train, xs_cpcint_train,
             xs_cpintfip_train, xs_cpintfref_train],
            ys_train,
            batch_size=batch_size,
            epochs=epochs,
            validation_split=0.1,
            callbacks=[early_stop],
            verbose=1,
        )

        # Evaluate on test fold
        mean_ic, ics = compute_ic(
            model, xs_pitch_test, xs_cpint_test, xs_cpcint_test,
            xs_cpintfip_test, xs_cpintfref_test, ys_test,
        )
        melody_ics = compute_ic_per_melody(
            model, test_melodies, window_size, pitch_to_idx,
            interval_offset, interval_vocab_size,
        )

        print(f"\nFold {fold + 1} mean IC: {mean_ic:.3f} bits")
        all_ics.extend(ics)
        fold_results.append({
            "fold": fold + 1,
            "mean_ic": mean_ic,
            "train_size": len(train_melodies),
            "test_size": len(test_melodies),
            "melody_ics": melody_ics,
        })

    # Export fold assignments for IDyOM replication
    df_folds = pd.DataFrame(fold_assignments)
    if export_folds_path:
        df_folds.to_csv(export_folds_path, index=False)
        print(f"\nFold assignments saved to: {export_folds_path}")

    # Summary
    overall_mean_ic = np.mean(all_ics)
    overall_std_ic = np.std(all_ics)

    print(f"\n{'='*60}")
    print(f"OVERALL RESULTS ({k}-fold cross-validation)")
    print(f"{'='*60}")
    print(f"Mean IC: {overall_mean_ic:.3f} bits")
    print(f"Std IC:  {overall_std_ic:.3f} bits")
    for r in fold_results:
        print(f"  Fold {r['fold']}: {r['mean_ic']:.3f} bits ({r['test_size']} test melodies)")

    return {
        "overall_mean_ic": overall_mean_ic,
        "overall_std_ic": overall_std_ic,
        "all_ics": all_ics,
        "fold_results": fold_results,
        "fold_assignments": df_folds,
    }

## Essen

In [11]:
# essen_melodies = pd.read_csv("essen_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()
# essen_melody_ids = pd.read_csv("essen_unique_melodies.csv")["melody_id"].tolist()

# results = run_kfold(
#     essen_melodies, melody_ids=essen_melody_ids, k=10,
#     window_size=10, epochs=35,
#     export_folds_path="essen_fold_assignments_viewpoints.csv",
# )

## Meertens

In [ ]:
# meertens_melodies = pd.read_csv("meertens_unique_melodies.csv")["pitch"].apply(ast.literal_eval).tolist()
# meertens_melody_ids = pd.read_csv("meertens_unique_melodies.csv")["melody_id"].tolist()

# results = run_kfold(
#     melodies=meertens_melodies,
#     melody_ids=meertens_melody_ids,
#     k=10,
#     window_size=10,
#     epochs=35,
#     export_folds_path="meertens_fold_assignments_viewpoints.csv",
# )